# run_grid_overlay: live 8x8 grid overlay + square-center error (M1)

Needs the real camera and the board with its 4 corner markers. Not executed in CI.
Deliverable: a screenshot of the grid on the live image and the square-center pixel error.

**TODO**
- [ ] run `run_calibrate.ipynb` first (intrinsics), or skip undistortion
- [ ] ground truth for the error metric: click the 64 square centers (or a subset, e.g. the
      4 inner corners of each quadrant) on a saved frame and save them to
      `calibration/clicked_centers.json`
- [ ] report mean / max error over several camera placements / lighting setups

In [ ]:
%run ./calibration.ipynb
%run ./homography.ipynb
%cd ..
import json
from pathlib import Path

import cv2
import numpy as np

cam_cfg = load_config("configs/camera.yaml")
geom = load_board_geometry()
intr = load_intrinsics() if default_intrinsics_path().exists() else None
print("intrinsics:", "loaded" if intr else "missing (no undistortion)")

## Live overlay (press q to quit, s to save a frame)

In [ ]:
cap = cv2.VideoCapture(cam_cfg.camera.device)
last_H = None
try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if intr:
            frame = undistort(frame, intr["K"], intr["dist"])
        try:
            last_H, _ = compute_homography(frame, geom, cam_cfg.warp)
        except RuntimeError as e:
            cv2.putText(frame, str(e)[:60], (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        shown = draw_grid_overlay(frame, last_H, geom, cam_cfg.warp) if last_H is not None else frame
        cv2.imshow("grid overlay", shown)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("s"):
            cv2.imwrite("calibration/overlay.png", shown)
            cv2.imwrite("calibration/frame.png", frame)
        if key == ord("q"):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()

## Square-center error against clicked ground truth

In [ ]:
# TODO M1: create calibration/clicked_centers.json as {"e4": [x, y], ...} from calibration/frame.png
clicked = json.loads(Path("calibration/clicked_centers.json").read_text())
pred = grid_model_image(last_H, geom, cam_cfg.warp)["centers"]
sq_idx = [chess.parse_square(s) for s in clicked]
err = square_center_pixel_error(pred[sq_idx], np.array(list(clicked.values())))
print("square-center error (px):", err)